#   Cleaning and processing the data

the first steps in cleaning the data involve importing the libraries we'll use, these include:

1. pandas
2. re

In [6]:
import pandas as pd
import re

following this, we assign the dataframe of the raw file to a variable 'raw'

In [7]:

raw = pd.read_excel(r"C:\Users\sidlu\Mauritius_Tourism_Analytics\data\raw\2023 tourist arrivals\Tourist_M_Dec23_160224.xlsx", sheet_name="Table 2", header=None)

we then explore the data and determine how to proceed with the cleanup

In [8]:
raw.head(10)

,0,1,2,3,4,5,6,7,8,9,...,69,70,71,72,73,74,75,76,77,78
0,Back to Table of Contents,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table 2 - Tourist arrivals by country of resid...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Country of residence,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Jan,NaN,NaN,Feb,NaN,NaN,Mar,NaN,NaN,...,NaN,Nov,NaN,NaN,Dec,NaN,NaN,Jan-Dec,NaN,NaN
5,NaN,Air,Sea,Total,Air,Sea,Total,Air,Sea,Total,...,Total,Air,Sea,Total,Air,Sea,Total,Air,Sea,Total
6,EUROPE,34981,60,35041,44289,18,44307,45747,43,45790,...,86774,87256,2879,90135,84969,4084,89053,823581,11244,834825
7,Austria,1503,0,1503,1601,0,1601,1307,0,1307,...,2228,2671,122,2793,2073,105,2178,19664,337,20001
8,Belgium,761,0,761,748,0,748,836,0,836,...,2443,1998,11,2009,1623,28,1651,19457,74,19531
9,Bulgaria,205,1,206,305,0,305,239,0,239,...,83,243,2,245,423,150,573,2882,168,3050


Following our observation, we determine that we need to create a new data frame with the necessary rows and columns.

To do this we first define the row and column variables we'll use and assign their values.

The ffill (forward fill) function was used in the case of merged cells, like for the months rows which were converted from [jan, null, null] to [jan, jan, jan]

In [9]:
year_row = raw.iloc[3, 1:].ffill()
year_row = year_row.apply(lambda x: re.match(r"\d+", str(x)).group() if pd.notna(x) else x)
month_row = raw.iloc[4, 1:].ffill()
type_row  = raw.iloc[5, 1:]

new_columns = ["Country"]
for y, m, t in zip(year_row, month_row, type_row):
    new_columns.append(f"{y}_{m}_{t}")

data = raw.iloc[5:129].copy()
data.columns = new_columns
data = data.reset_index(drop=True)

data["Country"] = data["Country"].astype(str).str.replace(r"\s*\d+$", "", regex=True).str.strip()

# Detect region header rows: they're written in ALL CAPS (e.g. "EUROPE"), unlike country names
is_region_header = data["Country"].str.isupper()

# Build Continent column by carrying the most recent region header down onto the rows below it
data["Continent"] = data["Country"].where(is_region_header).ffill()

data = data[~is_region_header].copy()

The melt function is the real hero here, it's what converts the table from a wide form into a long form

In [10]:
long_df = data.melt(id_vars=["Country", "Continent"], var_name="Year_Month_Type", value_name="value")

long_df[["Year", "Month", "Type"]] = long_df["Year_Month_Type"].str.rsplit("_", n=2, expand=True)
long_df = long_df.drop(columns="Year_Month_Type")

long_df = long_df[(long_df["Year"] == "2023") & (long_df["Month"] != "Jan-Dec")]

now we use the variable tidy to store the cleaned up dataframe, most importantly, we reverse the melt of the type column to obtain
3 distinct columns being: "sea:, "air" and "total"

In [11]:
tidy = long_df.pivot_table(
    index=["Continent", "Country", "Year", "Month"],
    columns="Type",
    values="value",
    aggfunc="first"
).reset_index()
tidy.columns.name = None

some final cleanup of unnecessary data, and we are ready to export the file into our processed data folder as a CSV!

In [12]:

tidy["Date"] = pd.to_datetime(tidy["Year"] + "-" + tidy["Month"], format="%Y-%b")


tidy=tidy.sort_values(["Continent","Country", "Date"]).reset_index(drop=True)

tidy = tidy[tidy['Country'].notna()]

tidy = tidy[~tidy['Country'].str.contains("Other", na=False)]
tidy = tidy[~tidy['Country'].str.contains("All", na=False)]


tidy.to_csv("C:\\Users\\sidlu\\Mauritius_Tourism_Analytics\\data\\processed\\2023_Tourist_Arrivals_Clean.CSV", index=False)


Here's how the final table looks like:

In [13]:
tidy.head(10)

,Continent,Country,Year,Month,Air,Sea,Total,Date
0,AFRICA,Algeria,2023,Jan,44,0,44,2023-01-01
1,AFRICA,Algeria,2023,Feb,13,0,13,2023-02-01
2,AFRICA,Algeria,2023,Mar,21,0,21,2023-03-01
3,AFRICA,Algeria,2023,Apr,9,0,9,2023-04-01
4,AFRICA,Algeria,2023,May,27,0,27,2023-05-01
5,AFRICA,Algeria,2023,Jun,27,0,27,2023-06-01
6,AFRICA,Algeria,2023,Jul,22,0,22,2023-07-01
7,AFRICA,Algeria,2023,Aug,41,0,41,2023-08-01
8,AFRICA,Algeria,2023,Sep,38,0,38,2023-09-01
9,AFRICA,Algeria,2023,Oct,20,0,20,2023-10-01
